In [1]:
from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [3]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
# Step 1 - Load all the papers as LangChain documents using the pypdfloader

import glob
from langchain_community.document_loaders import PyPDFLoader

file_paths = glob.glob('../files/*.pdf')
all_docs = []

def clean_doc(doc):
    # Encode to bytes ignoring errors, then decode back to str
    doc.page_content = doc.page_content.encode('utf-8', 'ignore').decode('utf-8', 'ignore')
    return doc

for file_path in file_paths:
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    clean_docs = []
    for doc in docs:
        clean_docs.append(clean_doc(doc)) 
    all_docs.extend(clean_docs)

print(len(all_docs))

70


In [5]:
# Step 2 - split the document into overlapping chunks that are more manageable

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(all_docs)

print(f"Split papers into {len(all_splits)} sub-documents.")

Split papers into 373 sub-documents.


In [6]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['1e547f35-1191-4bea-b6b6-802250bccc1f', '08c62004-5a4d-4d69-9ebf-b59be6023396', '4e9b1e68-732a-45e2-81b2-9f65e67be30e']


This completes the Indexing portion of the pipeline. At this point we have a query-able vector store containing the chunked contents of our blog post. Given a user question, we should ideally be able to return the snippets of the blog post that answer the question.

# Retrieval and Generation
RAG applications commonly work as follows:

    1. Retrieve: Given a user input, relevant splits are retrieved from storage using a Retriever.
    2. Generate: A model produces an answer using a prompt that includes both the question with the retrieved data

## RAG agents
One formulation of a RAG application is as a simple agent with a tool that retrieves information. We can assemble a minimal RAG agent by implementing a tool that wraps our vector store:

In [7]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [8]:
from langchain.agents import create_agent

tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a few research papers about human-robot interaction. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [17]:
query = (
    "How is the interaction between humans and robots modeled in a majority of these papers?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How is the interaction between humans and robots modeled in a majority of these papers?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (1ae858d6-9501-4e85-89ad-f9e90467f1bf)
 Call ID: 1ae858d6-9501-4e85-89ad-f9e90467f1bf
  Args:
    query: How is the interaction between humans and robots modeled in a majority of these papers?
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'TCPDF 6.10.0 (http://www.tcpdf.org); modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Certified by IEEE PDFExpress at March 27, 2025 03:28:23', 'creationdate': '2025-09-05T01:44:15+04:00', 'meeting starting date': '26 May 2025', 'moddate': '2025-09-11T22:23:14-04:00', 'ieee article id': '